El siguiente Notebook representa el flujo de trabajo en el framework Flex para efectuar el ataque conocido como "additive noise".

Primero se carga la base de datos a utilizar, en este caso Mnist.(para mayor información consultar la documentación oficial de la tecnología.)

In [1]:
from process_data import *
from copy import deepcopy

flex_dataset, server_id = load_and_preprocess_horizontal(dataname="mnist", trasnform=False, nodes=10)

torch.Size([60000, 28, 28])


A continuación, se define la arquitectura de los modelos locales de los clientes. Para el presente ejemplo se trabaja con modelos neuronales de pytorch.

Se utiliza el módulo networks_models, quien contiene una serie de modelos neuronales auxiliares de pytorch, para el trabajo con las bases de datos anteriormente mencionadas. Además se utiliza el módulo auxiliar networks_execution, que define la ejecución del entrenamiento y otros detalles de estos modelos.

Para establecer un modelo personalizado, ir a la documentación de Flex.

In [2]:
from networks_models import *
from networks_execution import *
from flex.pool import init_server_model
from flex.model import FlexModel

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.cuda.is_available()
    else "cpu"
)

net_config = ExecutionNetwork()


@init_server_model
def build_server_model():
    server_flex_model = FlexModel()
    criterion, model, optimizer = net_config.for_fd_server_model_config()
    server_flex_model["model"] = model.to(device)
    # Required to store this for later stages of the FL training process
    server_flex_model["criterion"] = criterion
    server_flex_model["optimizer_func"] = optimizer
    server_flex_model["optimizer_kwargs"] = {}
    return server_flex_model


A continuacion se define la arquitectura del modelo federado

In [3]:
from flex.pool import FlexPool

flex_pool = FlexPool.client_server_pool(
    fed_dataset=flex_dataset, server_id=server_id, init_func=build_server_model
)

clients = flex_pool.clients
servers = flex_pool.servers
aggregators = flex_pool.aggregators

# Select clients
clients_per_round = 10
selected_test_clients_pool = clients.select(clients_per_round)
selected_test_clients = selected_test_clients_pool.clients

Se define la funcion para desplegar el modelo global de cada cliente

In [4]:
from flex.pool import deploy_server_model
import copy

@deploy_server_model
def copy_server_model_to_clients(server_flex_model: FlexModel):
    return copy.deepcopy(server_flex_model)

Se define las rondas de entrenamiento local del cliente.

In [5]:
def train(client_flex_model: FlexModel, client_data: Dataset):
    print(np.array(client_data.X_data).shape)
    train_dataset = client_data.to_torchvision_dataset(transform=mnist_transform())
    client_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)

    model = client_flex_model['model']
    model = model.to(device)

    client_flex_model["previous_model"] = deepcopy(
        model
    )
    optimizer = client_flex_model["optimizer_func"]
    criterion = client_flex_model["criterion"]

    net_config.train_network(local_epochs=1, criterion=criterion, optimizer=optimizer, momentum=0.9, lr=0.005,
                             trainloader=client_dataloader, testloader=None,
                             model=model)

    return client_flex_model

Se define la función destinada a afectar la agregación del modelo mediante ruido aditivo. Este ataque está diseñado para afectar principalmente al Diferencial de Privacida, ya que utiliza el propio ruido generado por el agregador.

In [6]:
from attack import param_manipulated_attacks as param_manipulated
from attack.DeSMP_variants import desmp_attack
from flexclash.model import model_poison_agregator
import random as ran
import torch

@model_poison_agregator
def desmp(list_of_weights: list):
    # Extraer configuración del ataque
    sigma = 0.1  # Ajustar según el nivel de DP en el experimento
    ganma = 0.5

    # Transformar pesos
    mod_dic = param_manipulated.adecuate_params(list_of_weights)

    sensitivity = torch.median(torch.stack([torch.norm(param) for param in mod_dic]))



    # Lista de clientes participantes
    agent_id_id = selected_test_clients.actor_ids

    # Definir clientes maliciosos
    num_corrupt = ran.sample(agent_id_id, 1)  # Simulación con un cliente malicioso

    # Aplicar ataque DeSMP
    attacked_updates = desmp_attack(mod_dic, sigma, sensitivity, ganma, num_corrupt, agent_id_id)

    # Revertir transformacion
    final_upd = param_manipulated.adecuate_params_reverse(attacked_updates, list_of_weights[0])

    return final_upd

Se efectúa la agregación del modelo federado

In [7]:
from flex.pool import collect_client_diff_weights_pt
from flexclash.pool import central_differential_privacy
from flex.pool import set_aggregated_diff_weights_pt

#pool.aggregators.map(collect_client_diff_weights_pt, selected_test_clients)
#pool.aggregators.map(central_differential_privacy)
#pool.aggregators.map(set_aggregated_diff_weights_pt, pool.servers)

Se evalúa el modelo federado

In [8]:
def evaluate_global_model(server_flex_model: FlexModel, test_data: Dataset):
    model = server_flex_model["model"]
    model.eval()
    test_loss = 0
    test_acc = 0
    total_count = 0
    model = model.to(device)

    criterion = server_flex_model["criterion"]
    # get test data as a torchvision object
    test_dataset = test_data.to_torchvision_dataset(transform=mnist_transform())
    test_dataloader = DataLoader(
        test_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=False
    )
    losses = []
    with torch.no_grad():
        for data, target in tqdm(test_dataloader):
            total_count += target.size(0)
            data, target = data.to(device), target.to(device)
            output = model(data)
            losses.append(criterion(output, target).item())
            pred = output.data.max(1, keepdim=True)[1]
            test_acc += pred.eq(target.data.view_as(pred)).long().cpu().sum().item()

    test_loss = sum(losses) / len(losses)
    test_acc /= total_count

    return test_loss, test_acc

Para limpiar los modelos en memoria. Opcional

In [9]:
def clean_up_models(client_model: FlexModel, _):
    import gc

    client_model.clear()
    gc.collect()

Se definen las rondas de entrenamiento del modelo federado. Este método engloba los anteriores. Además en este caso se describe en cada momento como se enctua el ataque.

In [10]:
def train_n_rounds(n_rounds=2, clients_per_round=10):
    flex_pool = FlexPool.client_server_pool(
    fed_dataset=flex_dataset, server_id=server_id, init_func=build_server_model
)
    for i in range(n_rounds):
        print(f"\nRunning round: {i + 1} of {n_rounds}")
        selected_clients_pool = flex_pool.clients.select(clients_per_round)
        selected_clients = selected_clients_pool.clients
        print("Selected clients:", len(selected_clients))
        print(f"Selected clients for this round: {len(selected_clients)}")
        # Deploy the server model to the selected clients
        flex_pool.servers.map(copy_server_model_to_clients, selected_clients)
        # Each selected client trains her model
        selected_clients.map(train)
        # The aggregador collects weights from the selected clients and aggregates them
        flex_pool.aggregators.map(collect_client_diff_weights_pt, selected_clients)
        flex_pool.aggregators.map(desmp)
        flex_pool.aggregators.map(central_differential_privacy)

        # The aggregator send its aggregated weights to the server
        flex_pool.aggregators.map(set_aggregated_diff_weights_pt, flex_pool.servers)
        metrics = flex_pool.servers.map(evaluate_global_model)
        loss, acc = metrics[0]
        print(f"Global accuracy Server: Test acc: {acc:.4f}, test loss: {loss:.4f}")

        # Optional
        selected_clients.map(clean_up_models)

In [11]:
train_n_rounds(n_rounds=1, clients_per_round=10)


Running round: 1 of 1
Selected clients: 10
Selected clients for this round: 10
(6000, 28, 28)


100%|██████████| 24/24 [00:02<00:00,  8.53it/s]


(6000, 28, 28)


100%|██████████| 24/24 [00:02<00:00,  9.15it/s]


(6000, 28, 28)


...

100%|██████████| 24/24 [00:02<00:00,  8.59it/s]


(6000, 28, 28)


100%|██████████| 24/24 [00:02<00:00,  8.58it/s]


(6000, 28, 28)


 46%|████▌     | 11/24 [00:01<00:01,  8.41it/s]

KeyboardInterrupt

